In [6]:
pairs = pd.read_csv("orthologpairsfinal.csv")
pairs["baroacc"] = pairs["barophilus"].str.split("|").str[1]
pairs["kodaacc"] = pairs["kodakarensis"].str.split("|").str[1]
pairs["barosp"] = pairs["barophilus"].str.startswith("sp|")
pairs["kodasp"] = pairs["kodakarensis"].str.startswith("sp|")
print(pd.crosstab(pairs["barosp"], pairs["kodasp"]))

kodasp  False  True 
barosp              
False    1043    391
True        0      1


In [8]:
%pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 47.6 MB/s  0:00:00.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [statsmodels] 5/6 [statsmodels]
Note: you may need to restart the kernel to use updated packages.


In [10]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

table = pd.read_csv("tthermofinaltable.csv")         
fragile = "fragile100"  
d = table[["pI_x", fragile]].dropna().copy()
d.columns = ["pI", "fragile"]
d["fragile"] = d["fragile"].astype(int)
d["pI2"] = d["pI"] ** 2
print(len(d), "proteins")
def fit(df):
    return smf.logit("fragile ~ pI + pI2", data=df).fit(disp=0)
model = fit(d)
b1, b2 = model.params["pI"], model.params["pI2"]
c = -b1 / (2 * b2)
print(model.summary().tables[1])
print(f"\nc = {c:.2f}   (b2 = {b2:.3f}, p = {model.pvalues['pI2']:.2g}; b2 must be > 0 for U shape)")

1225 proteins
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      8.5421      1.293      6.609      0.000       6.009      11.075
pI            -2.4783      0.361     -6.862      0.000      -3.186      -1.770
pI2            0.1574      0.024      6.578      0.000       0.111       0.204

c = 7.87   (b2 = 0.157, p = 4.8e-11; b2 must be > 0 for U shape)


In [11]:
rng = np.random.default_rng(2026)
cs, not_u = [], 0
for _ in range(1000):
    s = d.sample(len(d), replace=True, random_state=rng)
    try:
        m = fit(s)
    except Exception:
        not_u += 1
        continue
    if m.params["pI2"] > 0:
        cs.append(-m.params["pI"] / (2 * m.params["pI2"]))
    else:
        not_u += 1
lo, hi = np.percentile(cs, [2.5, 97.5])
print(f"c 95% CI: {lo:.2f} to {hi:.2f}   ({not_u}/1000 resamples failed or weren't U-shaped)")

c 95% CI: 7.65 to 8.16   (0/1000 resamples failed or weren't U-shaped)


In [12]:
extremerate, middlerate = 0.607, 0.289
negligible = 0.05 * (extremerate - middlerate)

def p_hat(x):
    x = np.atleast_1d(np.asarray(x, dtype=float))
    return model.predict(pd.DataFrame({"pI": x, "pI2": x ** 2})).to_numpy()

u_med = np.median(np.abs(d["pI"] - c))
steps = np.linspace(0, 3, 3001)
rise = p_hat(c + u_med + steps) - p_hat(c + u_med)[0]
assert (rise >= negligible).any(), "curve never rises that much; pick smaller negligible"
delta = steps[np.argmax(rise >= negligible)]
print(f"median |pI - c| = {u_med:.2f}   negligible = {negligible:.3f}   delta = {delta:.3f} pH units")

median |pI - c| = 1.83   negligible = 0.016   delta = 0.119 pH units
